# darkcomp GPU benchmark: batched Gaia epoch-astrometry orbit search

Measures the throughput (sources per second) of the batched search pipeline
(`darkcomp.batch.run_pipeline`: single-star fit -> profile-likelihood periodogram -> bounded L-BFGS refinement)
on CPU vs GPU, **after** JIT warm-up, using mock sources built on the real scan patterns of the 12 Gaia DR4
pre-release sources.

Timing protocol: every configuration is run once untimed (compiles + allocates), then `n_rep` timed runs
with `jax.block_until_ready` on the outputs; the median is reported. Inputs are placed on the target device
beforehand so host-device transfer is not timed.

Workflow being benchmarked: the batched search is the **candidate filter** for every source; posterior sampling (NUTS)
is run afterwards only on the few interesting objects, so it is timed here on a single real candidate, not per star.

**Note on GPUs:** the likelihood runs in float64 by default. Consumer/T4/L4-class GPUs execute float64 at 1/32 to 1/64 of
their float32 rate; A100/H100 at 1/2. Check the printed GPU name before interpreting the numbers. A float32
variant of the *search* stage is included as an experiment (the final fit should stay float64).

In [ ]:
# Runtime > Change runtime type > GPU. Then:
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
!pip -q install -U "jax[cuda12]" numpyro jaxopt astropy pandas
import jax; print(jax.__version__, jax.devices())

## Get the code
The repository is private. Either (a) paste a GitHub personal access token with `repo` read scope, or
(b) upload a zip of the repository via the Files pane and set `REPO_DIR` accordingly.

In [ ]:
import os, getpass
REPO = "Hanyuan0908/astrometry_jax"
REPO_DIR = "/content/astrometry_jax"
if not os.path.exists(REPO_DIR):
    token = getpass.getpass("GitHub token (leave empty if you uploaded a zip): ")
    if token:
        !git clone -q https://{token}@github.com/{REPO}.git {REPO_DIR}
    else:
        raise SystemExit("Upload the repository zip, unzip it to REPO_DIR, and re-run this cell.")
sys.path.insert(0, REPO_DIR); os.chdir(REPO_DIR)
import darkcomp  # enables float64
from darkcomp import dr4, epoch as E, batch, mock
print("darkcomp imported from", darkcomp.__file__)

In [ ]:
import numpy as np
# Real DR4 pre-release scan patterns (12 sources) as templates
eps, df = dr4.load_prerelease("data/dr4_prerelease/gaia-dr4-prerelease-epoch-astrometry_2026-06-26.zip")
# precise scan patterns only (median per-CCD error < 0.5 mas); the G~19-20 sources have mas-level errors and would dominate the non-detections
templates = [e for e in eps.values() if np.median(e.err[e.mask > 0]) < 0.5]
print(len(templates), "templates")
print({dr4.PRERELEASE_SOURCES[s][0] or dr4.PRERELEASE_SOURCES[s][1]: e.n for s, e in eps.items()})

In [ ]:
import time, numpy as np, jax, jax.numpy as jnp

def make_batch(n_sources, seed=0):
    rng = np.random.default_rng(seed)
    truth = mock.random_population(rng, n_sources)
    sims = mock.simulate(rng, templates, truth)
    return batch.stack_epochs(sims), truth

def bench(dd, device, periods, n_rep=3, chunk=256, refine_chunk=64, label=""):
    """Median post-warm-up wall time of the full pipeline on `device`."""
    dd_dev = jax.device_put(dd, device); periods = jax.device_put(jnp.asarray(periods), device)
    n = dd["w"].shape[0]
    with jax.default_device(device):
        out = batch.run_pipeline(dd_dev, periods, chunk=chunk, refine_chunk=refine_chunk)   # warm-up: compile
        jax.block_until_ready(out["period"])
        times = []
        for _ in range(n_rep):
            t0 = time.perf_counter()
            out = batch.run_pipeline(dd_dev, periods, chunk=chunk, refine_chunk=refine_chunk)
            jax.block_until_ready(out["period"])
            times.append(time.perf_counter() - t0)
    t = float(np.median(times))
    print(f"{label:>10s} {str(device):>22s}  N={n:5d}  median {t:7.2f} s  ->  {n/t:8.1f} sources/s   (runs: {np.round(times,2)})")
    return out, t

PERIODS = np.geomspace(20, 3000, 300)   # 300 periods x 5 ecc x 8 phases = 12,000 trial orbits per source
cpu = jax.devices("cpu")[0]
gpu = jax.devices("gpu")[0] if any(d.platform == "gpu" for d in jax.devices()) else None
print("cpu:", cpu, " gpu:", gpu)

## CPU vs GPU at fixed batch size

In [ ]:
dd, truth = make_batch(128)
out_cpu, t_cpu = bench(dd, cpu, PERIODS, label="CPU")
if gpu is not None:
    out_gpu, t_gpu = bench(dd, gpu, PERIODS, label="GPU")
    print(f"speed-up GPU/CPU: {t_cpu/t_gpu:.1f}x")
    # sanity: same answers on both devices
    print("max |dP/P| between devices:", float(jnp.max(jnp.abs(out_gpu["period"]/out_cpu["period"] - 1))))

## Throughput vs batch size on the GPU
Throughput should rise with N until the device is saturated; `chunk` bounds memory (chunk x n_obs x 9 x 8 bytes).

In [ ]:
if gpu is not None:
    for n in (64, 256, 1024):
        dd_n, _ = make_batch(n)
        bench(dd_n, gpu, PERIODS, n_rep=2, chunk=1024, refine_chunk=256, label=f"GPU N={n}")

## Did the search recover the injected orbits?

In [ ]:
out = out_gpu if gpu is not None else out_cpu
P = np.asarray(out["period"]); sig = np.asarray(out["significance"]); a0 = np.asarray(out["a0_mas"])
rel = np.abs(P / truth["period"] - 1); det = sig > 5
print(f"detected (>5 sigma): {det.sum()}/{len(P)}; period within 5%: {(rel[det] < 0.05).sum()} of detected")
print("a0 of non-detections [mas]:", np.sort(truth['a0_mas'][~det]).round(2))
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].loglog(truth["period"], P, ".", c="0.5"); ax[0].loglog(truth["period"][det], P[det], "k."); ax[0].plot([20, 3000], [20, 3000], "r-", lw=0.8)
ax[0].set_xlabel("true P [d]"); ax[0].set_ylabel("recovered P [d]")
ax[1].loglog(truth["a0_mas"], sig, "k."); ax[1].axhline(5, c="r", lw=0.8); ax[1].set_xlabel("true a0 [mas]"); ax[1].set_ylabel("significance")
plt.tight_layout(); plt.show()

## Mixed precision: float32 periodogram, float64 refinement
`search_dtype=jnp.float32` runs only the expensive grid stage in float32. Checked on CPU: identical detection set, periodogram log-likelihoods within ~1 of float64 (peaks are 10^3 high), refinement unchanged because it stays float64. On GPUs with slow float64 (T4/L4) this is where the speed-up comes from.

In [ ]:
if gpu is not None:
    dd_g = jax.device_put(dd, gpu); P_g = jax.device_put(jnp.asarray(PERIODS), gpu)
    with jax.default_device(gpu):
        res = {}
        for label, kw in (("float64", {}), ("mixed f32 search", {"search_dtype": jnp.float32})):
            out = batch.run_pipeline(dd_g, P_g, chunk=1024, refine_chunk=256, **kw); jax.block_until_ready(out["period"])
            ts = []
            for _ in range(3):
                t0 = time.perf_counter(); out = batch.run_pipeline(dd_g, P_g, chunk=1024, refine_chunk=256, **kw); jax.block_until_ready(out["period"]); ts.append(time.perf_counter() - t0)
            res[label] = out
            print(f"{label:>18s}: median {np.median(ts):6.2f} s for {dd['w'].shape[0]} sources -> {dd['w'].shape[0]/np.median(ts):7.1f} sources/s")
        same_det = np.array_equal(np.asarray(res["float64"]["significance"]) > 5, np.asarray(res["mixed f32 search"]["significance"]) > 5)
        dP = np.abs(np.asarray(res["mixed f32 search"]["period"]) / np.asarray(res["float64"]["period"]) - 1)
        print("same detection set:", same_det, "| periods agree <1%:", (dP < 0.01).sum(), "/", len(dP))
        # pure float32 end to end, for reference only (refinement is less reliable in float32)
        dd32 = batch.as_dtype(dd_g, jnp.float32)
        out = batch.run_pipeline(dd32, jnp.asarray(PERIODS, jnp.float32), chunk=1024, refine_chunk=256); jax.block_until_ready(out["period"])
        t0 = time.perf_counter(); out = batch.run_pipeline(dd32, jnp.asarray(PERIODS, jnp.float32), chunk=1024, refine_chunk=256); jax.block_until_ready(out["period"])
        print(f"{'pure float32':>18s}: {time.perf_counter()-t0:6.2f} s (reference only)")

## Follow-up stage: NUTS posterior for one candidate (Gaia BH3, real data)
This is **not** part of the per-source pipeline. After the batch search ranks candidates by significance and companion mass, NUTS is run only on the interesting ones. Timed once on CPU and GPU for reference; a single chain is latency-bound, so do not expect a GPU gain here.

In [ ]:
import pandas as pd
from darkcomp import data as D, nuts, fit, likelihood as L
ep = E.EpochAstrometry.from_bh3_table("data/gaia_bh3/epochast.dat")
rv = pd.read_csv("data/gaia_bh3/epochrv.dat", names="transit_id jd rv err".split(), sep=r"\s+", comment="#")
rvd = D.RVData.from_arrays(rv.jd, rv.rv, rv.err)
d = ep.as_jax(); ll, be, bp = E.period_search(d, np.geomspace(50, 8000, 600)); k = int(np.argmax(np.asarray(ll)))
r = E.refine_profile(d, np.geomspace(50, 8000, 600)[k], float(be[k]), float(bp[k])); lin = r["linear"]
m2 = fit.m2_from_a0(r["a0_mas"], lin["parallax"], r["period"], 0.76)
theta0 = np.array([r["period"], r["ecc"], 0.76, m2, r["inc"], r["Omega"], r["omega"], lin["parallax"], lin["d_ra_mas"], lin["d_dec_mas"], lin["pmra"], lin["pmdec"], r["t_peri"], float(np.average(rv.rv, weights=1/rv.err**2))])
data = E.build_data(ep, rvd, m_star_prior=(0.76, 0.05))
for dev in [cpu] + ([gpu] if gpu is not None else []):
    with jax.default_device(dev):
        dd_ = jax.device_put(data, dev)
        nuts.run_nuts(E.log_prob, dd_, theta0, num_warmup=50, num_samples=50)   # warm-up / compile
        t0 = time.perf_counter(); theta, mcmc = nuts.run_nuts(E.log_prob, dd_, theta0, num_warmup=800, num_samples=1000); dt = time.perf_counter() - t0
        s = nuts.summarize(theta)
        print(f"{str(dev):>22s}: NUTS 800+1000 in {dt:.1f} s;  M_BH = {s['m_2'][0]:.2f} +{s['m_2'][1]:.2f} -{s['m_2'][2]:.2f} Msun  (paper 32.70 +- 0.82)")
print("Note: a single NUTS chain is latency-bound; GPUs win only when many chains/sources run in one vmapped call.")